# AI vs AI: A Chatbot Conversation (GPT vs Gemini)

This notebook sets up a fun little experiment: **two LLMs talking to each other**.

- **GPT (`gpt-4.1-mini`)** is given a system prompt that makes it **argumentative and snarky** — it disagrees with everything.
- **Gemini (`gemini-2.5-flash-lite`)** is given a system prompt that makes it **polite and agreeable** — it tries to find common ground and calm GPT down.

We then make them talk to each other for several rounds:
1. GPT sends a message.
2. Gemini responds to GPT's message.
3. We feed GPT's latest message back to GPT (as the "user" turn) so it keeps arguing, and feed Gemini's latest reply into itself the same way.
4. Repeat, building up two parallel conversation histories.

**What's new / fixed compared to the original snippet:**
- Loads API keys securely from a `.env` file using `python-dotenv` instead of hardcoding them.
- Installs and imports the correct SDKs (`openai` for GPT, and Google's OpenAI-compatible endpoint for Gemini).
- Initializes proper API clients (`openai` client and a `gemini` client pointed at Google's OpenAI-compatible endpoint).
- Adds error handling and sanity checks for missing API keys.
- Adds clear markdown explanations between each logical step.
- Wraps the display logic so it also works outside Jupyter (falls back to `print` if `IPython` isn't available).

Let's build it step by step.


## 1. Install dependencies

We need:
- `openai` — official SDK, used both for GPT and (via a custom `base_url`) for Gemini, since Google exposes an OpenAI-compatible endpoint.
- `python-dotenv` — to load API keys from a `.env` file instead of hardcoding them.


In [ ]:
# Install required packages (safe to re-run; will just confirm already-satisfied)
!pip install openai python-dotenv -q


## 2. Imports and API key setup

We load environment variables from a `.env` file in the same directory as this notebook.

Your `.env` file should look like this:

```
OPENAI_API_KEY=sk-...your-openai-key...
GOOGLE_API_KEY=AIza...your-google-key...
```

We never print the raw key values — only whether they were found — to avoid accidentally leaking secrets in notebook output.


In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

# Try to import IPython display helpers; fall back to print() if unavailable
# (e.g. if this notebook is ever run as a plain script)
try:
    from IPython.display import display, Markdown
    IN_NOTEBOOK = True
except ImportError:
    IN_NOTEBOOK = False
    def display(x):
        print(x)
    def Markdown(x):
        return x

# Load variables from a .env file into the environment
load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

# Basic sanity checks so failures are obvious and early, not buried in a stack trace later
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set - please add OPENAI_API_KEY to your .env file")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:8]}")
else:
    print("Google API Key not set - please add GOOGLE_API_KEY to your .env file")


## 3. Initialize the API clients

- `openai_client` talks to OpenAI's API directly for `gpt-4.1-mini`.
- `gemini_client` also uses the `openai` SDK, but pointed at **Google's OpenAI-compatible endpoint**, so we can call `gemini-2.5-flash-lite` with the exact same `chat.completions.create` interface.

This is why the original code could call `gemini.chat.completions.create(...)` — Google's Gemini API supports the OpenAI SDK format via a compatibility layer.


In [ ]:
openai_client = OpenAI(api_key=openai_api_key)

gemini_client = OpenAI(
    api_key=google_api_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)


## 4. Define models and personalities (system prompts)

- **GPT** is instructed to be argumentative, snarky, and to disagree with everything.
- **Gemini** is instructed to be polite, agreeable, and to de-escalate.


In [ ]:
# Using cheap/lightweight model variants to keep costs minimal
gpt_model = "gpt-4.1-mini"
gemini_model = "gemini-2.5-flash-lite"

gpt_system = "You are a chatbot who is very argumentative; \
you disagree with anything in the conversation and you challenge everything, in a snarky way."

gemini_system = "You are a very polite, courteous chatbot. You try to agree with \
everything the other person says, or find common ground. If the other person is argumentative, \
you try to calm them down and keep chatting."


## 5. Conversation history

We keep two parallel lists:
- `gpt_messages` — everything GPT has said.
- `gemini_messages` — everything Gemini has said.

Each starts with an opening line ("Hi there" / "Hi").


In [ ]:
gpt_messages = ["Hi there"]
gemini_messages = ["Hi"]


## 6. Helper functions to call each model

### `call_gpt()`
Builds the message list from GPT's point of view:
- GPT's own past messages are tagged `"assistant"`.
- Gemini's past messages are tagged `"user"` (since, from GPT's perspective, Gemini is "the user" it's replying to).

### `call_gemini()`
Same idea in reverse:
- GPT's messages are tagged `"user"` (Gemini is replying to GPT).
- Gemini's own past messages are tagged `"assistant"`.
- We append one extra `"user"` turn at the end with GPT's most recent message, since the `zip()` loop only covers messages up to the shorter list's length (this matches the original logic, kept intentionally).


In [ ]:
def call_gpt():
    messages = [{"role": "system", "content": gpt_system}]
    for gpt, gemini in zip(gpt_messages, gemini_messages):
        messages.append({"role": "assistant", "content": gpt})
        messages.append({"role": "user", "content": gemini})
    response = openai_client.chat.completions.create(model=gpt_model, messages=messages)
    return response.choices[0].message.content


def call_gemini():
    messages = [{"role": "system", "content": gemini_system}]
    for gpt, gemini_message in zip(gpt_messages, gemini_messages):
        messages.append({"role": "user", "content": gpt})
        messages.append({"role": "assistant", "content": gemini_message})
    messages.append({"role": "user", "content": gpt_messages[-1]})
    response = gemini_client.chat.completions.create(model=gemini_model, messages=messages)
    return response.choices[0].message.content


## 7. Quick sanity check (optional)

Before running the full loop, let's make one call to each function to confirm the API keys and clients work.


In [ ]:
# Uncomment to test a single call to each model before running the full conversation loop
# print("GPT says:", call_gpt())
# print("Gemini says:", call_gemini())


## 8. Run the conversation

We reset the histories to their starting points, print the opening lines, then loop for 5 rounds:

1. GPT replies (based on the conversation so far).
2. We display and store GPT's reply.
3. Gemini replies (based on GPT's latest message).
4. We display and store Gemini's reply.

Each round grows both conversation histories, so later replies are informed by everything said before.


In [ ]:
# Reset conversation histories to their starting point before running the loop
gpt_messages = ["Hi there"]
gemini_messages = ["Hi"]

display(Markdown(f"### GPT:\n{gpt_messages[0]}\n"))
display(Markdown(f"### Gemini:\n{gemini_messages[0]}\n"))

NUM_ROUNDS = 5

for i in range(NUM_ROUNDS):
    gpt_next = call_gpt()
    display(Markdown(f"### GPT:\n{gpt_next}\n"))
    gpt_messages.append(gpt_next)

    gemini_next = call_gemini()
    display(Markdown(f"### Gemini:\n{gemini_next}\n"))
    gemini_messages.append(gemini_next)


## 9. Notes & next steps

- Try changing `gpt_system` / `gemini_system` to give the bots different personalities (e.g. two overly-polite bots, or two argumentative bots).
- Increase `NUM_ROUNDS` for a longer conversation (keep an eye on API costs).
- You could extend this to 3+ models by adding another `messages` list and another `call_...()` function following the same pattern.
- Remember: never commit your `.env` file or API keys to version control.
